<a href="https://colab.research.google.com/github/katze-a0/building_detection_classification/blob/main/tif_to_jpg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#1 run
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#2 run
!pip install rasterio

In [ ]:
!unzip /content/drive/MyDrive/Nakkhu_Satellite_Clip.zip -d /content/input_folder

unzip:  cannot find or open /content/drive/MyDrive/Nakkhu_Satellite_Clip.zip, /content/drive/MyDrive/Nakkhu_Satellite_Clip.zip.zip or /content/drive/MyDrive/Nakkhu_Satellite_Clip.zip.ZIP.


In [ ]:
!cp -r /content/drive/MyDrive/Minor/Nakkhu_Satellite_Clip -d /content/input_folder

cp: cannot stat '/content/drive/MyDrive/Minor/Nakkhu_Satellite_Clip': No such file or directory


In [ ]:
#3 run - 01-41 samma ekaichoti run gardeu ..folder at once
import rasterio
import numpy as np
from PIL import Image
import os
import glob

# 2. Setup Folders
# We use 'input_folder' to match where your !cp command sent the files
input_folder = '/content/input_folder/Nakkhu_Satellite_Clip'
output_folder = '/content/output_folder'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# 3. Get list of all .tif files and sort them for consistency
tif_files = sorted(glob.glob(os.path.join(input_folder, "*.tif")))

# 4. Select your batch
# Based on your previous logic, adjust these numbers for each batch
# Example for your requested range:
current_batch = tif_files[2952:3034]

print(f"✅ Found {len(tif_files)} total TIFFs.")
print(f"🚀 Processing {len(current_batch)} files in this batch...")

for input_tif in current_batch:
    try:
        # Generate output filename (change .tif to .jpg)
        base_name = os.path.basename(input_tif)
        output_jpg = os.path.join(output_folder, base_name.replace('.tif', '.jpg'))

        with rasterio.open(input_tif) as src:
            # Read first three bands
            data = src.read([1, 2, 3])

            # Reorder (Bands, H, W) -> (H, W, Bands)
            data = np.transpose(data, (1, 2, 0))

            # Normalize data to 0-255
            data_reshaped = data.astype('float32')
            for i in range(3):
                band_min = data_reshaped[:,:,i].min()
                band_max = data_reshaped[:,:,i].max()
                if band_max - band_min != 0:
                    data_reshaped[:,:,i] = (data_reshaped[:,:,i] - band_min) * (255 / (band_max - band_min))

            final_img = data_reshaped.astype('uint8')

            # Save as JPG
            img = Image.fromarray(final_img)
            img.save(output_jpg, "JPEG", quality=95)

            print(f" Processed: {base_name}")

    except Exception as e:
        print(f"Error processing {input_tif}: {e}")

print("\nAll done! Check your output folder.")

✅ Found 1 total TIFFs.
🚀 Processing 0 files in this batch...

All done! Check your output folder.


In [ ]:
#not necessary---just coordinate thaha paune matra ho
import rasterio
from rasterio.warp import transform

def get_tif_coordinates(file_path):
    with rasterio.open(file_path) as dataset:
        # Read the dataset's coordinate reference system (CRS)
        crs = dataset.crs
        print(f"CRS: {crs}")

        # Get the bounding box in the file's native projection
        bounds = dataset.bounds

        # Transform the bounds to WGS84 (Lat/Lon) if it's in a different projection (like UTM)
        # We take the corners: (left, bottom) and (right, top)
        longitudes, latitudes = transform(
            crs,
            'EPSG:4326',
            [bounds.left, bounds.right],
            [bounds.bottom, bounds.top]
        )

        print("-" * 30)
        print(f"Longitude Range: {longitudes[0]:.6f} to {longitudes[1]:.6f}")
        print(f"Latitude Range:  {latitudes[0]:.6f} to {latitudes[1]:.6f}")
        print("-" * 30)

        # Calculate the center point
        center_lon = (longitudes[0] + longitudes[1]) / 2
        center_lat = (latitudes[0] + latitudes[1]) / 2
        print(f"Center Point: Lat {center_lat:.6f}, Lon {center_lon:.6f}")

# Usage: Replace 'your_file.tif' with your uploaded file path
get_tif_coordinates('//content/input_folder/Nakkhu_Satellite_Clip/Nakkhu_Actual_Image_45_01.tif')

CRS: EPSG:3857
------------------------------
Longitude Range: 85.292126 to 85.293505
Latitude Range:  27.595310 to 27.596533
------------------------------
Center Point: Lat 27.595921, Lon 85.292816


In [ ]:
# 1. Zip the entire output folder
!zip -r Nakkhu_JPG_Tiles_Batch2.zip /content/output_folder

# 2. Download the zip file to your computer
from google.colab import files
files.download('Nakkhu_JPG_Tiles_Batch2.zip')

updating: content/output_folder/ (stored 0%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import csv
import json
from osgeo import gdal, osr

tif_folder = "/content/input_folder/Nakkhu_Satellite_Clip"       # path to your .tif files
output_csv = "tile_index.csv"
output_html = "tile_map.html"

def get_tile_bounds(tif_path):
    ds = gdal.Open(tif_path)
    if not ds:
        return None
    gt = ds.GetGeoTransform()
    cols, rows = ds.RasterXSize, ds.RasterYSize

    # corners in native projection
    x_min = gt[0]
    y_max = gt[3]
    x_max = gt[0] + cols * gt[1]
    y_min = gt[3] + rows * gt[5]

    # reproject to WGS84 (lat/lon)
    srs = osr.SpatialReference()
    srs.ImportFromWkt(ds.GetProjection())
    wgs84 = osr.SpatialReference()
    wgs84.ImportFromEPSG(4326)
    transform = osr.CoordinateTransformation(srs, wgs84)

    lon_min, lat_min, _ = transform.TransformPoint(x_min, y_min)
    lon_max, lat_max, _ = transform.TransformPoint(x_max, y_max)
    ds = None
    return lat_min, lat_max, lon_min, lon_max

tiles = []
for fname in sorted(os.listdir(tif_folder)):
    if fname.lower().endswith(".tif"):
        tif_path = os.path.join(tif_folder, fname)
        jpg_name = os.path.splitext(fname)[0] + ".jpg"
        bounds = get_tile_bounds(tif_path)
        if bounds:
            lat_min, lat_max, lon_min, lon_max = bounds
            tiles.append({
                "tif": fname,
                "jpg": jpg_name,
                "lat_min": round(lat_min, 6),
                "lat_max": round(lat_max, 6),
                "lon_min": round(lon_min, 6),
                "lon_max": round(lon_max, 6),
                "center_lat": round((lat_min + lat_max) / 2, 6),
                "center_lon": round((lon_min + lon_max) / 2, 6),
            })

# Save CSV
with open(output_csv, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=tiles[0].keys())
    writer.writeheader()
    writer.writerows(tiles)
print(f"Saved {len(tiles)} tiles to {output_csv}")

# Save interactive HTML map
geojson_features = [{"type":"Feature","properties":{"tif":t["tif"],"jpg":t["jpg"]},"geometry":{"type":"Polygon","coordinates":[[[t["lon_min"],t["lat_min"]],[t["lon_max"],t["lat_min"]],[t["lon_max"],t["lat_max"]],[t["lon_min"],t["lat_max"]],[t["lon_min"],t["lat_min"]]]]}} for t in tiles]
geojson = {"type": "FeatureCollection", "features": geojson_features}
center_lat = tiles[len(tiles)//2]["center_lat"]
center_lon = tiles[len(tiles)//2]["center_lon"]
html = f"""<!DOCTYPE html><html><head>
<link rel="stylesheet" href="https://unpkg.com/leaflet/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet/dist/leaflet.js"></script>
<style>body{{margin:0}}#map{{height:100vh}}</style>
</head><body><div id="map"></div><script>
var map = L.map('map').setView([{center_lat},{center_lon}], 12);
L.tileLayer('https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png').addTo(map);
L.geoJSON({json.dumps(geojson)}, {{
  style: {{color:'red',weight:1,fillOpacity:0.1}},
  onEachFeature: function(f,layer){{layer.bindPopup('<b>TIF:</b> '+f.properties.tif+'<br><b>JPG:</b> '+f.properties.jpg);}}
}}).addTo(map);
</script></body></html>"""

with open(output_html, "w") as f:
    f.write(html)
print(f"Open {output_html} in your browser")

Saved 1 tiles to tile_index.csv
Open tile_map.html in your browser


/usr/local/lib/python3.12/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [ ]:
from pyproj import Transformer

transformer = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

lon_min, lat_min = transformer.transform(3203761, 9494559)
lon_max, lat_max = transformer.transform(3206636, 9500001)

print(f"ROI_LAT_MIN = {lat_min}")
print(f"ROI_LAT_MAX = {lat_max}")
print(f"ROI_LON_MIN = {lon_min}")
print(f"ROI_LON_MAX = {lon_max}")

ROI_LAT_MIN = 64.56440851011094
ROI_LAT_MAX = 64.58539690991587
ROI_LON_MIN = 28.77987472966042
ROI_LON_MAX = 28.805701294078858


In [ ]:
import pandas as pd

df = pd.read_csv("tile_index.csv")

# Define your ROI bounding box (edit these!)
ROI_LAT_MIN = 85.292126
ROI_LAT_MAX = 85.328
ROI_LON_MIN = 28.77987472966042
ROI_LON_MAX = 28.805701294078858

mask = (
    (df["lat_max"] >= ROI_LAT_MIN) &
    (df["lat_min"] <= ROI_LAT_MAX) &
    (df["lon_max"] >= ROI_LON_MIN) &
    (df["lon_min"] <= ROI_LON_MAX)
)

selected = df[mask]
print(f"Found {len(selected)} tiles in your ROI:")
print(selected[["tif", "jpg", "center_lat", "center_lon"]])

# Save list of JPGs to annotate
selected["jpg"].to_csv("tiles_to_annotate.csv", index=False)

Found 0 tiles in your ROI:
Empty DataFrame
Columns: [tif, jpg, center_lat, center_lon]
Index: []


In [ ]:
from pyproj import Transformer

def convert_to_utm45n(lat, lon):
    # EPSG:4326 = WGS 84 (Degrees)
    # EPSG:32645 = WGS 84 / UTM zone 45N (Meters)
    # always_xy=True returns (Easting, Northing)
    transformer = Transformer.from_crs("epsg:4326", "epsg:32645", always_xy=True)

    easting, northing = transformer.transform(lon, lat)
    return easting, northing

# Example: Chapagaun Station
lat_chapagaun = 27.59306
lon_chapagaun = 85.37889

e, n = convert_to_utm45n(lat_chapagaun, lon_chapagaun)

print(f"Location: Chapagaun")
print(f"UTM 45N Easting (X):  {e:.3f} m")
print(f"UTM 45N Northing (Y): {n:.3f} m")


Location: Chapagaun
UTM 45N Easting (X):  340001.335 m
UTM 45N Northing (Y): 3053173.472 m


In [ ]:
#!/usr/bin/env python3
"""
Convert a single TIF to JPG and print its location info.

Usage:
    python tif_to_jpg_single.py input.tif
    python tif_to_jpg_single.py input.tif output.jpg
"""

pip install pillow rasterio numpy


import sys
import os
import numpy as np
from pathlib import Path

# ── auto-install deps ─────────────────────────────────────────────────────────
def install_if_missing():
    import importlib, subprocess
    for pkg, imp in [("Pillow", "PIL"), ("rasterio", "rasterio"), ("numpy", "numpy")]:
        try:
            importlib.import_module(imp)
        except ImportError:
            print(f"Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

install_if_missing()

from PIL import Image
import rasterio
from rasterio.transform import xy

# ── inputs ────────────────────────────────────────────────────────────────────
if len(sys.argv) < 2:
    print("Usage: python tif_to_jpg_single.py input.tif [output.jpg]")
    sys.exit(1)

tif_path = sys.argv[1]
jpg_path = sys.argv[2] if len(sys.argv) >= 3 else str(Path(tif_path).with_suffix(".jpg"))

# ── 1. Print geo/location info ────────────────────────────────────────────────
print(f"\n{'═'*55}")
print(f"  LOCATION INFO: {Path(tif_path).name}")
print(f"{'═'*55}")

with rasterio.open(tif_path) as src:
    t      = src.transform
    crs    = src.crs
    bounds = src.bounds
    W, H   = src.width, src.height

    # CRS
    if crs:
        epsg = crs.to_epsg()
        print(f"  CRS          : {'EPSG:' + str(epsg) if epsg else crs.to_proj4()}")
        print(f"  Type         : {'Geographic (lat/lon)' if crs.is_geographic else 'Projected (meters/units)'}")
    else:
        print("  CRS          : ⚠  None — not a GeoTIFF (no location data)")

    print(f"  Image size   : {W} × {H} px,  {src.count} band(s)")

    # The affine transform origin IS the top-left corner
    print(f"\n  Origin (top-left corner):")
    print(f"    X  =  {t.c}   ← {'longitude' if crs and crs.is_geographic else 'easting'}")
    print(f"    Y  =  {t.f}   ← {'latitude ' if crs and crs.is_geographic else 'northing'}")

    print(f"\n  Pixel size:")
    print(f"    width   = {abs(t.a)}")
    print(f"    height  = {abs(t.e)}")

    print(f"\n  Bounding box:")
    print(f"    West  (left)  = {bounds.left}")
    print(f"    East  (right) = {bounds.right}")
    print(f"    South (bot)   = {bounds.bottom}")
    print(f"    North (top)   = {bounds.top}")

    label = "lon, lat" if crs and crs.is_geographic else "x, y"
    print(f"\n  Corner coordinates  ({label}):")
    for name, (row, col) in [
        ("top_left    ", (0,     0    )),
        ("top_right   ", (0,     W - 1)),
        ("bottom_left ", (H - 1, 0    )),
        ("bottom_right", (H - 1, W - 1)),
    ]:
        x, y = xy(t, row, col)
        print(f"    {name} = ({round(x, 7)},  {round(y, 7)})")

print(f"{'═'*55}\n")

# ── 2. Convert TIF → JPG ──────────────────────────────────────────────────────
with rasterio.open(tif_path) as src:
    data  = src.read()
    count = src.count

    def stretch(arr):
        """Stretch any dtype to 0-255 uint8."""
        a = arr.astype(np.float32)
        lo, hi = a.min(), a.max()
        return ((a - lo) / (hi - lo) * 255).astype(np.uint8) if hi > lo else a.astype(np.uint8)

    if count == 1:
        img = Image.fromarray(stretch(data[0]), mode="L")

    elif count == 3:
        rgb = np.stack([stretch(data[i]) for i in range(3)], axis=-1)
        img = Image.fromarray(rgb, mode="RGB")

    else:  # 4+ bands — composite alpha onto white
        rgba = np.stack([stretch(data[i]) for i in range(4)], axis=-1)
        fg   = Image.fromarray(rgba, mode="RGBA")
        img  = Image.new("RGB", fg.size, (255, 255, 255))
        img.paste(fg, mask=fg.split()[3])

img.save(jpg_path, "JPEG", quality=95, optimize=True)
print(f"✓  Saved: {jpg_path}  ({os.path.getsize(jpg_path) / 1024:.1f} KB)")

SyntaxError: invalid syntax (3857364456.py, line 10)

In [ ]:
import rasterio
from PIL import Image
import numpy as np

with rasterio.open('/content/Nakkhu_base_1_1_in.tif') as src:
    # 1. Read the bands
    data = src.read([1, 2, 3])

    # 2. Reshape to (Height, Width, Channels)
    data = data.transpose(1, 2, 0)

    # 3. Handle the Float32 (<f4) error
    # If your data is 0-1, scale it. If it's already 0-255 but just stored as floats,
    # just do the .astype('uint8')
    if data.max() <= 1.0:
        data = (data * 255).astype('uint8')
    else:
        data = data.astype('uint8')

    # 4. Save
    img = Image.fromarray(data)
    img.save('output.jpg', "JPEG")



In [ ]:
##advanced
import rasterio
from PIL import Image
import numpy as np
import glob
import os

# 1. Automatically find all .tif files in the current directory
# Change path if files are in a specific folder (e.g., '/content/drive/MyDrive/*.tif')
tile_index = glob.glob("*.tif")

if not tile_index:
    print("No .tif files found. Please check your file path.")
else:
    print(f"Found {len(tile_index)} files. Starting processing...\n")
    print("-" * 50)

    for file_path in tile_index:
        try:
            with rasterio.open(file_path) as src:
                # --- EXTRACT SPATIAL METADATA ---
                crs = src.crs
                bounds = src.bounds # (left, bottom, right, top)
                transform = src.transform

                print(f"FILE: {file_path}")
                print(f"LOCATION (Bounds): {bounds}")
                print(f"CRS: {crs}")

                # --- PROCESS IMAGE DATA ---
                # Read first 3 bands (RGB)
                data = src.read([1, 2, 3])

                # Reshape from (Bands, Height, Width) to (Height, Width, Bands)
                data = data.transpose(1, 2, 0)

                # Fix the float32 (<f4) / DataType error for PIL
                # If values are 0-1 (float), scale to 0-255. Otherwise just cast.
                if data.dtype != 'uint8':
                    if data.max() <= 1.1: # Threshold for float-normalized data
                        data = (data * 255).astype('uint8')
                    else:
                        data = data.astype('uint8')

                # --- SAVE OUTPUT ---
                output_jpg = file_path.replace('.tif', '.jpg')
                img = Image.fromarray(data)
                img.save(output_jpg, "JPEG", quality=95)

                # Optional: Create a World File (.jgw) to keep the JPG georeferenced
                world_file = output_jpg.replace('.jpg', '.jgw')
                with open(world_file, "w") as f:
                    f.write(f"{transform.a}\n{transform.d}\n{transform.b}\n{transform.e}\n{transform.c}\n{transform.f}")

                print(f"RESULT: Successfully converted to {output_jpg}")
                print("-" * 50)

        except Exception as e:
            print(f"ERROR processing {file_path}: {e}")
            print("-" * 50)

Found 2 files. Starting processing...

--------------------------------------------------
FILE: Nakkhu_base_1_1_in.tif
LOCATION (Bounds): BoundingBox(left=9495215.0507, bottom=3206714.9074, right=9495727.0507, top=3207226.9074)
CRS: EPSG:3857
RESULT: Successfully converted to Nakkhu_base_1_1_in.jpg
--------------------------------------------------
FILE: Nakkhu_base_1_1.tif
LOCATION (Bounds): BoundingBox(left=9495215.0507, bottom=3206970.9074, right=9495471.0507, top=3207226.9074)
CRS: EPSG:3857
RESULT: Successfully converted to Nakkhu_base_1_1.jpg
--------------------------------------------------


In [ ]:
##check tif metadata- hand model tif
import rasterio

def get_tif_metadata(file_path):
    with rasterio.open(file_path) as src:
        print(f"--- Metadata for: {file_path} ---")

        # Core Metadata
        print(f"Width:  {src.width}")
        print(f"Height: {src.height}")
        print(f"Bands:  {src.count}")
        print(f"Bounds: {src.bounds}")

        # Coordinate Reference System (CRS)
        print(f"\n--- CRS Details ---")
        if src.crs:
            print(f"CRS: {src.crs.to_string()}")
            # Specifically check if it is Geographic (Lat/Lon) or Projected (Meters)
            print(f"Is Projected: {src.crs.is_projected}")
            print(f"Is Geographic: {src.crs.is_geographic}")
            print(f"Units: {src.crs.linear_units}")
        else:
            print("CRS is not defined in this file.")

        # Affine Transform (Pixel Resolution)
        # transform[0] is pixel width, transform[4] is pixel height (usually negative)
        print(f"\n--- Transform (Affine) ---")
        print(src.transform)
        print(f"Pixel Resolution: {src.res[0]} x {src.res[1]}")

# Run the function
# get_tif_metadata('path_to_your_hand_model.tif')